# Populace quickstart — United States

Load the published Populace US population and run tax-benefit analysis in a few lines.

**Data**: [`policyengine/populace-us`](https://huggingface.co/datasets/policyengine/populace-us) — a weighted synthetic population of 57,240 households calibrated to thousands of administrative targets (IRS SOI, CBO, Census, SSA, CMS and more), built entirely from primary public sources. Every release ships its build manifest, calibration diagnostics, and reform validation next to the data; per-target fit is browsable on the [calibration dashboard](https://calibration-diagnostics.vercel.app/populace).

**Engine**: [`policyengine-us`](https://github.com/PolicyEngine/policyengine-us) computes taxes and benefits on the population.

Runs on a free Colab CPU runtime in a few minutes.

In [1]:
%pip install -q policyengine-us huggingface_hub

## Load the population

The registry's root file tracks the latest published release (`latest.json` records the exact release id and manifests).

In [2]:
from huggingface_hub import hf_hub_download

h5_path = hf_hub_download(
    repo_id="policyengine/populace-us",
    filename="populace_us_2024.h5",
    repo_type="dataset",
)
h5_path

'/Users/maxghenis/.cache/huggingface/hub/datasets--policyengine--populace-us/snapshots/053baf6cf56aaf1160e2f1bfe7631c6924d46b2e/populace_us_2024.h5'

In [3]:
from policyengine_us import Microsimulation
from policyengine_us.data import USSingleYearDataset

sim = Microsimulation(dataset=USSingleYearDataset(file_path=h5_path))

## National aggregates

Weighted totals for 2025. Dollar levels are calibrated to the most recent published administrative facts (e.g. IRS SOI); see the [calibration dashboard](https://calibration-diagnostics.vercel.app/populace) for per-target fit.

In [4]:
year = 2025

population_m = sim.calculate("age", year).weights.sum() / 1e6
agi_t = sim.calculate("adjusted_gross_income", year).sum() / 1e12
income_tax_t = sim.calculate("income_tax", year).sum() / 1e12
benefits_t = sim.calculate("household_benefits", year).sum() / 1e12

print(f"population            {population_m:8.1f}M")
print(f"adjusted gross income ${agi_t:7.2f}T")
print(f"federal income tax    ${income_tax_t:7.2f}T")
print(f"household benefits    ${benefits_t:7.2f}T")

population               343.2M
adjusted gross income $  16.01T
federal income tax    $   2.15T
household benefits    $   1.99T


## Poverty

In [5]:
in_poverty = sim.calculate("in_poverty", year, map_to="person")
print(f"SPM poverty rate {year}: {in_poverty.mean() * 100:.1f}%")

SPM poverty rate 2025: 13.7%


## Score a reform

Raise the child tax credit base from \$2,200 to \$3,000 per child in 2026 and measure the federal income-tax revenue change.

In [6]:
from policyengine_core.reforms import Reform

reform = Reform.from_dict(
    {"gov.irs.credits.ctc.amount.base[0].amount": {"2026-01-01.2026-12-31": 3000}},
    country_id="us",
)

baseline_tax = sim.calculate("income_tax", 2026).sum()
reformed = Microsimulation(
    dataset=USSingleYearDataset(file_path=h5_path), reform=reform
)
reform_tax = reformed.calculate("income_tax", 2026).sum()

print(f"CTC $2,200 -> $3,000 (2026): federal cost ${(baseline_tax - reform_tax) / 1e9:.1f}B")

CTC $2,200 -> $3,000 (2026): federal cost $32.8B


## Next

- Browse every calibration target and each release's diagnostics: [calibration dashboard](https://calibration-diagnostics.vercel.app/populace)
- The stack and design charter: [github.com/PolicyEngine/populace](https://github.com/PolicyEngine/populace)
- The L0 dataset-reduction paper behind the 57k-household file: [populace.dev/papers/l0](https://populace.dev/papers/l0)